# 01 — Data audit: duplicates across splits, balanced-set provenance, multi-source bias

**Paper sections fed:** III-A (Dataset), V-D (Limitations). **CPU only** (no GPU quota needed).

1. Exact duplicates (MD5 of file bytes **and** of decoded pixels) within/across train, val, test and across classes.
2. Near-duplicates: perceptual hash (pHash, 64-bit) candidates, **confirmed** by pixel correlation at 128×128,
   because pHash alone gives many false positives on chest radiographs (they all look alike at 8×8 DCT scale).
   A visual grid of the closest cross-split pairs is produced for **manual confirmation** — report only confirmed pairs.
3. Balanced dataset provenance: are its val/test folders byte-identical to the original? Which balanced-train files
   are untouched originals vs. augmented copies, and do any augmented copies derive from **val/test** images (leakage)?
4. Multi-source bias: filename families, file format, resolution, colour mode and intensity statistics per class, plus two
   **shortcut probes** — a classifier on file metadata only, and one on a 32×32 thumbnail with the central lung field
   blanked. High test macro-F1 from either means the label is recoverable without looking at lung pathology.

In [ ]:
import os, re, io, json, hashlib, itertools, warnings
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
try:
    import imagehash
except ImportError:
    os.system('pip install -q imagehash'); import imagehash

ORIGINAL_ROOT = '/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis'
BALANCED_ROOT = '/kaggle/input/datasets/malihamostafa/balanceddatasetctbp'   # set None to skip section 3
OUT = '/kaggle/working/audit'; os.makedirs(OUT, exist_ok=True)
CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']
SPLITS = ['train', 'val', 'test']
PHASH_MAX_HAMMING = 8        # candidate threshold (of 64 bits); generous on purpose
CORR_CONFIRM = 0.90          # Pearson r of HIGH-PASS 128x128 images (global chest shape removed) needed to confirm
CORR_DERIVED = 0.50          # looser threshold for tracing rotated/jittered balanced-set copies to a source
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
# After inspecting balanced_embedding_leakage_flagged.png, list val/test images CONFIRMED BY EYE to have an augmented
# copy in the balanced train set, e.g. ['test/COVID19/COVID19(540).jpg']. They are added to the 'clean' exclusion list.
MANUAL_LEAKED = ['test/COVID19/COVID19(534).jpg']   # 3 flipped augmented copies in balanced train (aug_COVID19_837/9/103);
                                                    # confirm at full resolution before submission

In [ ]:
def index_root(root, tag, splits=SPLITS):
    rows = []
    for sp in splits:
        for c in CLASS_NAMES:
            d = os.path.join(root, sp, c)
            if not os.path.isdir(d): continue
            for fn in sorted(os.listdir(d)):
                if not fn.lower().endswith(IMG_EXTS): continue
                p = os.path.join(d, fn); raw = open(p, 'rb').read()
                im = Image.open(io.BytesIO(raw)); mode = im.mode; w, h = im.size
                g = im.convert('RGB').convert('L')
                a128 = np.asarray(g.resize((128, 128), Image.BILINEAR), dtype=np.float32)
                a = np.asarray(g, dtype=np.float32)
                rows.append(dict(source=tag, split=sp, cls=c, file=fn, path=p, ext=os.path.splitext(fn)[1].lower(),
                                 bytes=len(raw), md5_file=hashlib.md5(raw).hexdigest(),
                                 md5_pixels=hashlib.md5(np.asarray(g).tobytes()).hexdigest(),
                                 mode=mode, width=w, height=h, aspect=w / h, mean=a.mean(), std=a.std(),
                                 border_mean=np.concatenate([a[:max(1,h//20)].ravel(), a[-max(1,h//20):].ravel()]).mean(),
                                 phash=int(str(imagehash.phash(g)), 16), thumb=a128, hp=a128 - gaussian_filter(a128, 4)))
    return pd.DataFrame(rows)

D = index_root(ORIGINAL_ROOT, 'original')
print(D.groupby(['split', 'cls']).size().unstack(fill_value=0))

## 1. Exact duplicates

In [ ]:
def exact_dups(df, key):
    g = df.groupby(key)
    dup = df[df[key].isin(g.size()[g.size() > 1].index)].sort_values(key)
    rep = []
    for h, grp in dup.groupby(key):
        sps, cls = sorted(set(grp.split)), sorted(set(grp.cls))
        rep.append(dict(hash=h, n=len(grp), splits='|'.join(sps), classes='|'.join(cls),
                        cross_split=len(sps) > 1, label_conflict=len(cls) > 1,
                        files='; '.join(grp.split + '/' + grp.cls + '/' + grp.file)))
    return pd.DataFrame(rep)

for key in ['md5_file', 'md5_pixels']:
    E = exact_dups(D, key)
    print(f'\n[{key}] duplicate groups: {len(E)} | cross-split: {int(E.cross_split.sum()) if len(E) else 0}'
          f' | label conflicts: {int(E.label_conflict.sum()) if len(E) else 0}')
    if len(E): E.to_csv(f'{OUT}/exact_dups_{key}.csv', index=False); display(E.head(20))

## 2. Near-duplicates (pHash candidates → pixel-correlation confirmation)

In [ ]:
def popcount64(x):
    x = x - ((x >> 1) & 0x5555555555555555); x = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
    x = (x + (x >> 4)) & 0x0F0F0F0F0F0F0F0F; return (x * 0x0101010101010101) >> 56

def near_pairs(df_a, df_b=None, max_h=PHASH_MAX_HAMMING, chunk=512):
    same = df_b is None; df_b = df_a if same else df_b
    ha = df_a.phash.values.astype(np.uint64); hb = df_b.phash.values.astype(np.uint64)
    out = []
    for i0 in range(0, len(ha), chunk):
        d = popcount64(ha[i0:i0 + chunk, None] ^ hb[None, :]).astype(np.int16)
        ii, jj = np.where(d <= max_h)
        for i, j in zip(ii + i0, jj):
            if same and j <= i: continue
            out.append((i, j, int(d[i - i0, j])))
    return out

def corr(a, b):
    a = a - a.mean(); b = b - b.mean(); return float((a * b).sum() / (np.sqrt((a * a).sum() * (b * b).sum()) + 1e-8))

P = near_pairs(D)
rows = []
for i, j, h in P:
    a, b = D.iloc[i], D.iloc[j]
    rows.append(dict(i=i, j=j, hamming=h, r128=corr(a.thumb, b.thumb), r_hp=corr(a.hp, b.hp), split_a=a.split, split_b=b.split,
                     cls_a=a.cls, cls_b=b.cls, file_a=a.file, file_b=b.file))
N = pd.DataFrame(rows)
if len(N):
    N['cross_split'] = N.split_a != N.split_b
    N['confirmed'] = N.r_hp >= CORR_CONFIRM
    N['pair'] = N.apply(lambda r: '-'.join(sorted([r.split_a, r.split_b])), axis=1)
    print('pHash candidates:', len(N), '| confirmed (high-pass r>=%.2f):' % CORR_CONFIRM, int(N.confirmed.sum()),
          '| (plain 128px r>=0.95 would flag %d — shared anatomy, not duplicates)' % int((N.r128 >= .95).sum()))
    S = N[N.confirmed].groupby(['pair']).size().rename('confirmed_pairs').to_frame()
    S['label_conflicts'] = N[N.confirmed & (N.cls_a != N.cls_b)].groupby('pair').size()
    display(S.fillna(0).astype(int))
    # unique test images that have a confirmed near-duplicate in train/val
    te = N[N.confirmed & N.cross_split & ((N.split_a == 'test') | (N.split_b == 'test'))]
    te_idx = set(np.where(te.split_a == 'test', te.i, te.j))
    print(f'test images with a confirmed near-duplicate outside test: {len(te_idx)} / {(D.split=="test").sum()}')
    pd.Series(sorted(D.iloc[list(te_idx)].path)).to_csv(f'{OUT}/test_images_with_neardup_outside_test.csv', index=False)
    N.drop(columns=[]).to_csv(f'{OUT}/near_dup_candidates.csv', index=False)
else:
    print('no pHash candidates found')

In [ ]:
# Visual confirmation grid: the closest cross-split pairs (manual check — report only what you confirm by eye)
if len(N) and N.cross_split.any():
    top = N[N.cross_split].sort_values(['r_hp'], ascending=False).head(12)
    fig, ax = plt.subplots(len(top), 2, figsize=(5, 2.4 * len(top)))
    ax = np.atleast_2d(ax)
    for k, (_, r) in enumerate(top.iterrows()):
        for m, idx in enumerate([r.i, r.j]):
            row = D.iloc[idx]; ax[k, m].imshow(row.thumb, cmap='gray'); ax[k, m].axis('off')
            ax[k, m].set_title(f'{row.split}/{row.cls}\n{row.file[:22]}  r_hp={r.r_hp:.3f} h={r.hamming}', fontsize=7)
    plt.tight_layout(); plt.savefig(f'{OUT}/near_dup_cross_split_grid.png', dpi=130); plt.show()

## 3. Balanced dataset provenance and leakage

In [ ]:
if BALANCED_ROOT and os.path.isdir(BALANCED_ROOT):
    B = index_root(BALANCED_ROOT, 'balanced')
    print(B.groupby(['split', 'cls']).size().unstack(fill_value=0))
    # (a) val/test identical to original?
    for sp in ['val', 'test']:
        so = set(D[D.split == sp].md5_file); sb = set(B[B.split == sp].md5_file)
        print(f'{sp}: original {len(so)} | balanced {len(sb)} | identical sets: {so == sb} '
              f'| only-in-balanced {len(sb - so)} | only-in-original {len(so - sb)}')
    # (b) balanced train: untouched originals vs synthetic
    Bt = B[B.split == 'train'].copy()
    orig_md5 = {sp: set(D[D.split == sp].md5_pixels) for sp in SPLITS}
    for sp in SPLITS: Bt[f'exact_in_orig_{sp}'] = Bt.md5_pixels.isin(orig_md5[sp])
    print('\nbalanced-train files that are pixel-identical to an original image:')
    display(Bt.groupby('cls')[[f'exact_in_orig_{sp}' for sp in SPLITS]].sum().astype(int)
              .assign(total=Bt.groupby('cls').size()))
    # (c) synthetic copies: nearest original by pHash+correlation — which split did each come from?
    syn = Bt[~Bt[[f'exact_in_orig_{sp}' for sp in SPLITS]].any(axis=1)].reset_index(drop=True)
    if len(syn):
        cand = near_pairs(syn, D, max_h=16)
        best = {}
        for i, j, h in cand:
            r = corr(syn.hp[i], D.hp.iloc[j])
            if i not in best or r > best[i][1]: best[i] = (j, r, h)
        syn['nearest_split'] = [D.split.iloc[best[i][0]] if i in best else None for i in range(len(syn))]
        syn['nearest_r'] = [best[i][1] if i in best else np.nan for i in range(len(syn))]
        print(f'\nSynthetic balanced-train images: split of the closest original (high-pass r>={CORR_DERIVED} counted as derived):')
        syn['derived'] = syn.nearest_r >= CORR_DERIVED
        display(syn[syn.derived].groupby(['cls', 'nearest_split']).size().unstack(fill_value=0))
        print('synthetic images with no confident source (rotation can defeat matching; inspect if many):', int((~syn.derived).sum()))
        leak = syn[syn.derived & syn.nearest_split.isin(['val', 'test'])]
        print('>>> POSSIBLE LEAKAGE (synthetic train image derived from val/test):', len(leak))
        leak[['cls', 'file', 'nearest_split', 'nearest_r']].to_csv(f'{OUT}/balanced_leakage_candidates.csv', index=False)
else:
    print('BALANCED_ROOT not found — skipped')

### 3b. Targeted leakage check for augmented copies (rotation/flip search)
The provenance trace above fails for most augmented copies because rotation misaligns pixels. The only question that
matters for leakage is narrower: *does any augmented balanced-train image derive from a val/test image?* Each synthetic
image is matched against every val/test image under flips × rotations of −8…+8° (1° steps), using normalised
high-pass 64×64 correlation. **Positive control:** the same search against same-class train originals, where the true
source must be. If the control scores are high while val/test scores stay low, the check has power and finds no leakage.

In [ ]:
from scipy.ndimage import rotate as nd_rotate
def hp_vec(a, s=64, crop=0.8):
    x = np.asarray(Image.fromarray(np.clip(a, 0, 255).astype(np.uint8)).resize((s, s), Image.BILINEAR), np.float32)
    m = int(s * (1 - crop) / 2); x = x[m:s - m, m:s - m]
    x = x - gaussian_filter(x, 1.5); x = x - x.mean(); return (x / (np.linalg.norm(x) + 1e-8)).ravel()

def variants(a, angles=range(-8, 9)):
    for flip in (False, True):
        b = a[:, ::-1] if flip else a
        for ang in angles: yield nd_rotate(b, ang, reshape=False, order=1, mode='nearest')

def best_match(query, refs, ref_cls=None, query_cls=None):
    """max correlation of each query against any variant of any ref (optionally same class only)."""
    Q = np.stack([hp_vec(a) for a in query]); best = np.full(len(Q), -1.0); arg = np.full(len(Q), -1)
    for v_refs in zip(*[variants(a) for a in refs]):          # one (flip, angle) variant of all refs at a time
        Rm = np.stack([hp_vec(a) for a in v_refs]); S = Q @ Rm.T
        if ref_cls is not None: S = np.where(np.asarray(query_cls)[:, None] == np.asarray(ref_cls)[None, :], S, -1)
        j = S.argmax(1); v = S[np.arange(len(Q)), j]; upd = v > best; best[upd] = v[upd]; arg[upd] = j[upd]
    return best, arg

LEAKED = {}   # test/val relpath -> evidence; filled by 3b and 3d
if BALANCED_ROOT and os.path.isdir(BALANCED_ROOT) and len(syn):
    VT = D[D.split.isin(['val', 'test'])].reset_index(drop=True)
    s_vt, a_vt = best_match(list(syn.thumb), list(VT.thumb))
    TRm = D[(D.split == 'train') & D.cls.isin(syn.cls.unique())].reset_index(drop=True)
    s_tr, a_tr = best_match(list(syn.thumb), list(TRm.thumb), ref_cls=TRm.cls.values, query_cls=syn.cls.values)
    syn['score_valtest'], syn['score_train_source'] = s_vt, s_tr
    syn['closest_valtest'] = [VT.split[j] + '/' + VT.cls[j] + '/' + VT.file[j] for j in a_vt]
    print('positive control (true source in train): median score %.3f, 5th pct %.3f' % (np.median(s_tr), np.percentile(s_tr, 5)))
    print('val/test match:                          median score %.3f, 99th pct %.3f, max %.3f' % (np.median(s_vt), np.percentile(s_vt, 99), s_vt.max()))
    LEAK_T = max(0.8, float(np.percentile(s_tr, 5)))
    flag = syn[(syn.score_valtest >= LEAK_T) & (syn.score_valtest >= syn.score_train_source)]
    print(f'>>> synthetic images matching a val/test image at least as well as any train image (score ≥ {LEAK_T:.2f}): {len(flag)}')
    for _, r in flag[flag.score_valtest > flag.score_train_source].iterrows():
        LEAKED.setdefault(r.closest_valtest, f'augmented copy {r.file} in balanced train (pixel score {r.score_valtest:.2f})')
    syn[['cls', 'file', 'score_train_source', 'score_valtest', 'closest_valtest']].to_csv(f'{OUT}/balanced_synthetic_leakage_scores.csv', index=False)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(s_tr, 60, alpha=.6, label='vs train (true source, control)'); ax.hist(s_vt, 60, alpha=.6, label='vs val/test')
    ax.axvline(LEAK_T, c='k', ls=':'); ax.set_xlabel('best rotation/flip-searched high-pass correlation'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f'{OUT}/balanced_leakage_score_hist.png', dpi=130); plt.show()
    if len(flag):
        top = flag.sort_values('score_valtest', ascending=False).head(8)
        fig, ax = plt.subplots(len(top), 2, figsize=(5, 2.4 * len(top))); ax = np.atleast_2d(ax)
        for k, (_, r) in enumerate(top.iterrows()):
            ax[k, 0].imshow(r.thumb, cmap='gray'); ax[k, 0].set_title(f'balanced/train {r.file[:20]}', fontsize=7)
            vt = VT.iloc[a_vt[syn.index.get_loc(r.name)]]
            ax[k, 1].imshow(vt.thumb, cmap='gray'); ax[k, 1].set_title(f'{vt.split}/{vt.file[:20]} s={r.score_valtest:.2f}', fontsize=7)
            for a_ in ax[k]: a_.axis('off')
        plt.tight_layout(); plt.savefig(f'{OUT}/balanced_leakage_flagged.png', dpi=130); plt.show()

### 3d. Embedding-based leakage check (robust to rotation, scale, translation, brightness)
Each augmented balanced-train copy is compared with every val/test image by cosine similarity of ImageNet ResNet50
features (flip-searched). **Positive control:** similarity to the closest same-class train original, which must
contain the true source. Power = fraction of copies whose control similarity exceeds the threshold. A copy is flagged
when it is closer to a val/test image than to any train original **and** above the threshold. GPU ~2 min, CPU ~15 min.

In [ ]:
import torch, torchvision
from torchvision import transforms as T
if BALANCED_ROOT and os.path.isdir(BALANCED_ROOT) and len(syn):
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fx = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1); fx.fc = torch.nn.Identity()
    fx = fx.to(dev).eval()
    tf = T.Compose([T.Grayscale(3), T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
                    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    @torch.no_grad()
    def embed(paths, flip=False, bs=64):
        out = []
        for i in range(0, len(paths), bs):
            x = torch.stack([tf(Image.open(p).convert('RGB')) for p in paths[i:i + bs]]).to(dev)
            if flip: x = torch.flip(x, dims=[3])
            out.append(torch.nn.functional.normalize(fx(x), dim=1).cpu())
        return torch.cat(out).numpy()
    Eq = embed(list(syn.path))
    VT = D[D.split.isin(['val', 'test'])].reset_index(drop=True)
    TRm = D[(D.split == 'train') & D.cls.isin(syn.cls.unique())].reset_index(drop=True)
    def best_sim(refs, same_cls=False):
        S = np.maximum(Eq @ embed(list(refs.path)).T, Eq @ embed(list(refs.path), flip=True).T)
        if same_cls: S = np.where(np.asarray(syn.cls, dtype=object)[:, None] == np.asarray(refs.cls, dtype=object)[None, :], S, -1)
        return S.max(1), S.argmax(1)
    e_tr, _ = best_sim(TRm, same_cls=True); e_vt, j_vt = best_sim(VT)
    syn['emb_train_source'], syn['emb_valtest'] = e_tr, e_vt
    syn['emb_closest_valtest'] = [VT.split[j] + '/' + VT.cls[j] + '/' + VT.file[j] for j in j_vt]
    EMB_T = float(np.percentile(e_tr, 5))
    print(f'positive control: median {np.median(e_tr):.3f}; threshold = 5th pct = {EMB_T:.3f} '
          f'(=> power ≈ 95% for copies as similar to their source as typical)')
    print(f'val/test similarity: median {np.median(e_vt):.3f}, 99th pct {np.percentile(e_vt, 99):.3f}, max {e_vt.max():.3f}')
    fl = syn[(syn.emb_valtest >= EMB_T) & (syn.emb_valtest > syn.emb_train_source)].sort_values('emb_valtest', ascending=False)
    print(f'>>> embedding-flagged copies (closer to a val/test image than to any train original, ≥ threshold): {len(fl)} '
          f'-> {fl.emb_closest_valtest.nunique()} distinct val/test images')
    syn[['cls', 'file', 'emb_train_source', 'emb_valtest', 'emb_closest_valtest']].to_csv(f'{OUT}/balanced_embedding_leakage_scores.csv', index=False)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(e_tr, 80, alpha=.6, label='vs train (true source, control)'); ax.hist(e_vt, 80, alpha=.6, label='vs val/test')
    ax.axvline(EMB_T, c='k', ls=':'); ax.set_xlabel('max cosine similarity (ResNet50 features)'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f'{OUT}/balanced_embedding_leakage_hist.png', dpi=130); plt.show()
    if len(fl):
        top = fl.head(12); fig, ax = plt.subplots(len(top), 2, figsize=(5, 2.4 * len(top))); ax = np.atleast_2d(ax)
        for k, (_, r) in enumerate(top.iterrows()):
            vt = VT[(VT.split + '/' + VT.cls + '/' + VT.file) == r.emb_closest_valtest].iloc[0]
            ax[k, 0].imshow(r.thumb, cmap='gray'); ax[k, 0].set_title(f'balanced/train {r.file[:24]} (src {r.emb_train_source:.3f})', fontsize=7)
            ax[k, 1].imshow(vt.thumb, cmap='gray'); ax[k, 1].set_title(f'{r.emb_closest_valtest[:30]} sim={r.emb_valtest:.3f}', fontsize=7)
            for a_ in ax[k]: a_.axis('off')
        plt.tight_layout(); plt.savefig(f'{OUT}/balanced_embedding_leakage_flagged.png', dpi=130); plt.show()
        print('Confirm each pair by eye. Pairs that are visibly the same radiograph are leakage; keep only those in LEAKED.')
    sep = np.median(e_tr) - np.median(e_vt)
    if sep < 0.02:
        print(f'WARNING: control and val/test similarity medians differ by only {sep:.3f}; embeddings do not discriminate '
              'well here, so treat flags as weak candidates.')
    fl[['cls', 'file', 'emb_train_source', 'emb_valtest', 'emb_closest_valtest']].to_csv(f'{OUT}/balanced_embedding_flags_for_review.csv', index=False)
    print('Embedding flags are NOT auto-excluded: confirm by eye, then add confirmed paths to MANUAL_LEAKED (config cell) and re-run 3c.')
for p in MANUAL_LEAKED: LEAKED.setdefault(p, 'confirmed by visual inspection (embedding/pixel flag)')
print('val/test images with an augmented copy in balanced train (pixel-confirmed + manual):', len(LEAKED))
pd.Series(LEAKED, name='evidence', dtype=object).rename_axis('relpath').to_csv(f'{OUT}/balanced_leaked_valtest_images.csv')

### 3c. Test-set exclusion lists for a sensitivity analysis (used by notebook 03)
* `clean`  — drop repeated copies inside test (keep one per exact-duplicate group) and any test image with an exact or
  confirmed near-duplicate in train/val.
* `strict` — additionally drop test images with a cross-split pHash candidate (Hamming ≤ 4) and high-pass r ≥ 0.60,
  i.e. likely same-patient/follow-up images. Inspect the grid; these are not confirmed duplicates.

In [ ]:
rel = lambda r: f'{r.split}/{r.cls}/{r.file}'
excl = {}
for _, grp in D[D.split == 'test'].groupby('md5_pixels'):
    for _, r in grp.iloc[1:].iterrows(): excl[rel(r)] = 'repeat copy inside test (exact)'
outside = set(D[D.split != 'test'].md5_pixels)
for _, r in D[(D.split == 'test') & D.md5_pixels.isin(outside)].iterrows(): excl[rel(r)] = 'exact duplicate in train/val'
if len(N):
    for _, r in N[N.confirmed & N.cross_split].iterrows():
        for side, other in [('a', 'b'), ('b', 'a')]:
            if r[f'split_{side}'] == 'test':
                excl.setdefault(rel(D.iloc[r.i if side == 'a' else r.j]), f'confirmed near-duplicate in {r[f"split_{other}"]}')
for p, ev in (LEAKED.items() if 'LEAKED' in globals() else []):
    if p.startswith('test/'): excl.setdefault(p, 'leaked into balanced train: ' + ev)
clean = pd.DataFrame(sorted(excl.items()), columns=['relpath', 'reason']); clean['list'] = 'clean'
strict_extra = {}
if len(N):
    for _, r in N[N.cross_split & ~N.confirmed & (N.hamming <= 4) & (N.r_hp >= 0.60)].iterrows():
        for side, other in [('a', 'b'), ('b', 'a')]:
            if r[f'split_{side}'] == 'test':
                p = rel(D.iloc[r.i if side == 'a' else r.j])
                if p not in excl: strict_extra.setdefault(p, f'likely same patient in {r[f"split_{other}"]} (r_hp={r.r_hp:.2f}, h={r.hamming})')
strict = pd.DataFrame(sorted(strict_extra.items()), columns=['relpath', 'reason']); strict['list'] = 'strict_only'
EXC = pd.concat([clean, strict]); EXC.to_csv(f'{OUT}/test_exclusion_list.csv', index=False)
print(f'clean excludes {len(clean)} test images; strict excludes {len(clean) + len(strict)}')
with pd.option_context('display.max_colwidth', 100): display(EXC)

## 4. Multi-source bias

In [ ]:
def family(fn):
    s = os.path.splitext(fn)[0]
    s = re.sub(r'\d+', '#', s)            # digits -> '#'
    s = re.sub(r'#(?:[ _\-]#)+', '#', s)  # collapse runs
    return s

D['family'] = D.file.map(family)
fam = D.groupby(['cls', 'family']).size().rename('n').reset_index().sort_values(['cls', 'n'], ascending=[True, False])
print('Filename families per class (top 5 each):')
display(fam.groupby('cls').head(5))
print('\nFile format by class:'); display(pd.crosstab(D.cls, D.ext))
print('\nColour mode by class:'); display(pd.crosstab(D.cls, D['mode']))
print('\nResolution / intensity by class (median [IQR]):')
q = lambda s: f'{s.median():.0f} [{s.quantile(.25):.0f}–{s.quantile(.75):.0f}]'
display(D.groupby('cls').agg(width=('width', q), height=('height', q), aspect=('aspect', lambda s: f'{s.median():.2f}'),
                             mean_intensity=('mean', q), border_mean=('border_mean', q), kbytes=('bytes', lambda s: q(s / 1024))))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for c in CLASS_NAMES:
    d = D[D.cls == c]
    ax[0].scatter(d.width, d.height, s=4, alpha=.4, label=c); ax[1].hist(d['mean'], 40, alpha=.5, label=c)
    ax[2].hist(np.log10(d.bytes), 40, alpha=.5, label=c)
ax[0].set(xlabel='width', ylabel='height', title='resolution'); ax[1].set(title='mean intensity')
ax[2].set(title='log10 file size'); ax[0].legend(); plt.tight_layout(); plt.savefig(f'{OUT}/source_stats.png', dpi=130); plt.show()

In [ ]:
# Shortcut probes: train on TRAIN split, evaluate on TEST split (val too small).
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score

y = D.cls.map({c: i for i, c in enumerate(CLASS_NAMES)}).values
tr, te = (D.split == 'train').values, (D.split == 'test').values
meta = pd.get_dummies(D[['width', 'height', 'aspect', 'bytes', 'ext', 'mode']], columns=['ext', 'mode']).astype(float).values
rf = RandomForestClassifier(500, random_state=0, n_jobs=-1, class_weight='balanced').fit(meta[tr], y[tr])
pm = rf.predict(meta[te])

def masked_thumb(a, keep_border=0.2):          # blank the central lung field, keep outer frame
    t = np.asarray(Image.fromarray(a.astype(np.uint8)).resize((32, 32), Image.BILINEAR), dtype=np.float32) / 255.
    b = int(32 * keep_border); t[b:32 - b, b:32 - b] = 0; return t.ravel()
X = np.stack([masked_thumb(a) for a in D.thumb])
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight='balanced')).fit(X[tr], y[tr])
pt = lr.predict(X[te])
probe = pd.DataFrame([
    dict(probe='file metadata only (size, format, colour mode, bytes)', test_acc=accuracy_score(y[te], pm), test_macro_f1=f1_score(y[te], pm, average='macro')),
    dict(probe='32x32 thumbnail, central 60% blanked (border only)', test_acc=accuracy_score(y[te], pt), test_macro_f1=f1_score(y[te], pt, average='macro')),
    dict(probe='chance (stratified)', test_acc=float((np.bincount(y[te]) / te.sum()) @ (np.bincount(y[te]) / te.sum())), test_macro_f1=0.25)])
display(probe.round(4)); probe.to_csv(f'{OUT}/shortcut_probes.csv', index=False)
print('Interpretation: macro-F1 well above chance from metadata or image borders alone means class is confounded with '
      'acquisition source; report this as a limitation and as motivation for external validation.')

In [ ]:
# Save a compact summary for the paper
summary = dict(n_images=len(D), per_split=D.split.value_counts().to_dict())
for key in ['md5_file', 'md5_pixels']:
    E = exact_dups(D, key); summary[f'exact_dup_groups_{key}'] = len(E)
    summary[f'exact_dup_cross_split_{key}'] = int(E.cross_split.sum()) if len(E) else 0
if len(N):
    summary['near_dup_confirmed_pairs'] = int(N.confirmed.sum())
    summary['near_dup_confirmed_cross_split'] = int((N.confirmed & N.cross_split).sum())
summary['shortcut_probe_macro_f1'] = dict(zip(probe.probe, probe.test_macro_f1.round(4)))
json.dump(summary, open(f'{OUT}/audit_summary.json', 'w'), indent=1, default=str); print(json.dumps(summary, indent=1, default=str))
D.drop(columns=['thumb', 'hp']).to_csv(f'{OUT}/file_index.csv', index=False)